# RQ4 — Multimodal RoBERTa AES (Single-Task per Trait, No Grid Search)

Mirror dari `multimodal/gemma roberta gridsearch copy.ipynb` tanpa grid search.

**Hyperparameters tetap:** `lr=2e-5`, `dropout=0.2`, `epochs=15`.

Set `TARGET_TRAIT` ke `"clarity"` atau `"persuasive"` lalu jalankan seluruh notebook.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import cohen_kappa_score
from tqdm import tqdm
from bert_score import score
import gc
import random
import warnings
import os

warnings.filterwarnings("ignore")

def set_seed(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    try:
        if hasattr(torch, 'xpu') and torch.xpu.is_available():
            torch.xpu.manual_seed(seed)
            torch.xpu.manual_seed_all(seed)
    except ImportError:
        print("Warning: intel_extension_for_pytorch (ipex) tidak ditemukan.")
    torch.use_deterministic_algorithms(True, warn_only=True)

set_seed(42)

def get_device():
    if hasattr(torch, 'xpu') and torch.xpu.is_available():
        return torch.device("xpu")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = get_device()
print(f"Using Device: {DEVICE}")

c:\Users\gabyg\miniconda3\envs\skripsi\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using Device: xpu


## Konfigurasi

In [2]:
# === Pilih trait yang dilatih ===
TARGET_TRAIT = "clarity"   # "clarity" atau "persuasive"

GROUP_COL          = "graph"
MODEL_NAME         = "roberta-base"
MAX_LEN            = 512
BATCH_SIZE         = 2
ACCUMULATION_STEPS = 4              # Effective batch size = 8
UNFREEZE_LAYERS    = 6
DATA_FILE          = "../../short_gemma_bertscore.csv"

# Hyperparameters TETAP (no grid search)
LEARNING_RATE = 2e-5
DROPOUT_RATE  = 0.2
EPOCHS        = 15

BEST_MODEL_SAVE_PATH = f"best_roberta_{TARGET_TRAIT}_rq4.pth"
TRAIN_PRED_CSV       = f"roberta_train_{TARGET_TRAIT}_rq4.csv"
TEST_PRED_CSV        = f"roberta_test_{TARGET_TRAIT}_rq4.csv"

print(f"Target Trait : {TARGET_TRAIT.upper()}")
print(f"LR={LEARNING_RATE} | Drop={DROPOUT_RATE} | Epochs={EPOCHS}")

Target Trait : CLARITY
LR=2e-05 | Drop=0.2 | Epochs=15


## Prapemrosesan & BERTScore

In [3]:
def prepare_data(csv_path):
    df = pd.read_csv(csv_path).dropna(subset=['description']).reset_index(drop=True)
    if 'bertscore_f1' not in df.columns:
        print("\n[*] Menghitung BERTScore antara Esai dan Description...")
        cands = df['Essay'].fillna("").tolist()
        refs  = df['description'].fillna("").tolist()
        P, R, F1 = score(cands, refs, lang="en", verbose=True)
        df['bertscore_precision'] = P.numpy()
        df['bertscore_recall']    = R.numpy()
        df['bertscore_f1']        = F1.numpy()
        df.to_csv(csv_path, index=False)
        print(f"[*] Selesai! Data disimpan ke {csv_path}\n")
    return df

## Dataset (Single-Task with BERTScore)

In [4]:
class SingleTaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, trait):
        self.df         = df.reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.max_len    = max_len
        self.questions  = self.df['Question'].fillna("").values
        self.deplots    = self.df['description'].fillna("").values
        self.essays     = self.df['Essay'].fillna("").values
        self.bertscores = self.df['bertscore_f1'].fillna(0.0).values

        if trait == "clarity":
            self.targets = self.df['argument_clarity(ground_truth)'].values
        elif trait == "persuasive":
            self.targets = self.df['justifying_persuasiveness(ground_truth)'].values
        else:
            raise ValueError("TARGET_TRAIT harus 'clarity' atau 'persuasive'")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        suffix_text   = f"\n\nGraph Data: {self.deplots[index]} \n\nQuestion: {self.questions[index]}"
        suffix_tokens = self.tokenizer(suffix_text, add_special_tokens=False)['input_ids']

        prefix_text   = f"Essay: {self.essays[index]}"
        prefix_tokens = self.tokenizer(prefix_text, add_special_tokens=False)['input_ids']

        num_special_tokens = 2
        max_suffix_len = self.max_len - num_special_tokens
        if len(suffix_tokens) > max_suffix_len:
            suffix_tokens = suffix_tokens[:max_suffix_len]

        budget = self.max_len - len(suffix_tokens) - num_special_tokens
        prefix_tokens = prefix_tokens[:budget] if budget > 0 else []

        combined_tokens = prefix_tokens + suffix_tokens
        input_ids       = [self.tokenizer.cls_token_id] + combined_tokens + [self.tokenizer.sep_token_id]
        attention_mask  = [1] * len(input_ids)
        padding_length  = self.max_len - len(input_ids)
        if padding_length > 0:
            input_ids      += [self.tokenizer.pad_token_id] * padding_length
            attention_mask += [0] * padding_length

        return {
            'input_ids':      torch.tensor(input_ids,      dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'bertscore':      torch.tensor(self.bertscores[index], dtype=torch.float),
            'target':         torch.tensor(self.targets[index],    dtype=torch.float),
        }

## Model: RobertaAES (CLS token + BERTScore concat)

In [5]:
class RobertaAES(nn.Module):
    def __init__(self, model_name, dropout_rate=DROPOUT_RATE):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        self.drop = nn.Dropout(p=dropout_rate)

        for param in self.transformer.parameters():
            param.requires_grad = False
        for layer in self.transformer.encoder.layer[-UNFREEZE_LAYERS:]:
            for param in layer.parameters():
                param.requires_grad = True

        hidden_size = self.transformer.config.hidden_size
        self.score_head = nn.Linear(hidden_size + 1, 1)

    def forward(self, input_ids, attention_mask, bertscore):
        outputs        = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output  = outputs.last_hidden_state[:, 0, :]
        dropped_output = self.drop(pooled_output)
        combined       = torch.cat((dropped_output, bertscore), dim=1)
        return self.score_head(combined)

## Helper: Evaluate QWK

In [6]:
def evaluate_qwk(model, dataloader):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for b in dataloader:
            ids  = b['input_ids'].to(DEVICE)
            mask = b['attention_mask'].to(DEVICE)
            bs   = b['bertscore'].to(DEVICE).unsqueeze(1)
            tgt  = b['target'].to(DEVICE).unsqueeze(1)
            if DEVICE.type in ["cuda", "xpu"]:
                dtype = torch.bfloat16 if DEVICE.type == "xpu" else torch.float16
                with torch.autocast(device_type=DEVICE.type, dtype=dtype):
                    pred = model(ids, mask, bs)
            else:
                pred = model(ids, mask, bs)
            all_preds.extend(pred.float().cpu().numpy().flatten())
            all_targets.extend(tgt.float().cpu().numpy().flatten())

    pred_classes = np.round(np.array(all_preds)   * 2).astype(int)
    true_classes = np.round(np.array(all_targets) * 2).astype(int)
    return cohen_kappa_score(true_classes, pred_classes, weights='quadratic')

## STAGE 1 — Data Loading

In [7]:
df        = prepare_data(DATA_FILE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if os.path.exists('train_df_v2.csv') and os.path.exists('test_df_v2.csv'):
    print("Memuat train_df_v2.csv & test_df_v2.csv lokal ...")
    df_train = pd.read_csv('train_df_v2.csv')
    df_test  = pd.read_csv('test_df_v2.csv')
elif os.path.exists('../../multimodal/train_df_v2.csv') and os.path.exists('../../multimodal/test_df_v2.csv'):
    print("Memuat train_df_v2.csv & test_df_v2.csv dari ../../multimodal/ ...")
    df_train = pd.read_csv('../../multimodal/train_df_v2.csv')
    df_test  = pd.read_csv('../../multimodal/test_df_v2.csv')
elif os.path.exists('../../train_df.csv') and os.path.exists('../../test_df.csv'):
    print("Memuat ../../train_df.csv & ../../test_df.csv lalu mapping ke data Gemma...")
    df_train_ref = pd.read_csv('../../train_df.csv')
    df_test_ref  = pd.read_csv('../../test_df.csv')
    mapping_df = df[['graph', 'Essay', 'description', 'bertscore_precision', 'bertscore_recall', 'bertscore_f1']]
    df_train = pd.merge(df_train_ref, mapping_df, on=['graph', 'Essay'], how='left')
    df_test  = pd.merge(df_test_ref,  mapping_df, on=['graph', 'Essay'], how='left')
    df_train.to_csv('train_df_v2.csv', index=False)
    df_test.to_csv('test_df_v2.csv',  index=False)
    print(f"   Train: {df_train.shape} | Test: {df_test.shape}")
else:
    print("File split belum ada. Melakukan GroupShuffleSplit baru...")
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(df, groups=df[GROUP_COL]))
    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_test  = df.iloc[test_idx].reset_index(drop=True)
    df_train.to_csv('train_df_v2.csv', index=False)
    df_test.to_csv('test_df_v2.csv',  index=False)

test_dataset = SingleTaskDataset(df_test, tokenizer, MAX_LEN, TARGET_TRAIT)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nTotal: {len(df)} | Train: {len(df_train)} | Test: {len(df_test)}")

Memuat train_df_v2.csv & test_df_v2.csv dari ../../multimodal/ ...

Total: 1054 | Train: 867 | Test: 187


## STAGE 2 — Final Training (100% Train Set, Fixed Hyperparameters)

In [8]:
gc.collect()
if DEVICE.type == "xpu":    torch.xpu.empty_cache()
elif DEVICE.type == "cuda": torch.cuda.empty_cache()

set_seed(42)
g_final = torch.Generator(); g_final.manual_seed(42)

full_train_loader = DataLoader(
    SingleTaskDataset(df_train, tokenizer, MAX_LEN, TARGET_TRAIT),
    batch_size=BATCH_SIZE, shuffle=True, generator=g_final,
)

scaler          = torch.amp.GradScaler(DEVICE.type) if DEVICE.type in ["cuda", "xpu"] else None
final_model     = RobertaAES(MODEL_NAME, dropout_rate=DROPOUT_RATE).to(DEVICE)
final_optimizer = AdamW(
    filter(lambda p: p.requires_grad, final_model.parameters()),
    lr=LEARNING_RATE,
)
criterion = nn.MSELoss()

for epoch in range(EPOCHS):
    final_model.train()
    loop = tqdm(full_train_loader, desc=f"Final Epoch {epoch+1}/{EPOCHS}")
    for step, batch in enumerate(loop):
        ids     = batch['input_ids'].to(DEVICE)
        mask    = batch['attention_mask'].to(DEVICE)
        b_score = batch['bertscore'].to(DEVICE).unsqueeze(1)
        tgt     = batch['target'].to(DEVICE).unsqueeze(1)

        if scaler:
            dtype = torch.bfloat16 if DEVICE.type == "xpu" else torch.float16
            with torch.autocast(device_type=DEVICE.type, dtype=dtype):
                pred = final_model(ids, mask, b_score)
                loss = criterion(pred.float(), tgt) / ACCUMULATION_STEPS
            scaler.scale(loss).backward()
            if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(full_train_loader):
                scaler.step(final_optimizer); scaler.update(); final_optimizer.zero_grad()
        else:
            pred = final_model(ids, mask, b_score)
            loss = criterion(pred.float(), tgt) / ACCUMULATION_STEPS
            loss.backward()
            if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(full_train_loader):
                final_optimizer.step(); final_optimizer.zero_grad()

        loop.set_postfix({'Loss': f"{(loss.item() * ACCUMULATION_STEPS):.4f}"})

torch.save(final_model.state_dict(), BEST_MODEL_SAVE_PATH)
print(f"Final Model Tersimpan: {BEST_MODEL_SAVE_PATH}")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 294.45it/s, Materializing param=encoder.layer.11.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Final Epoch 15/15: 100%|██████████| 434/434 [02:41<00:00,  2.68it/s, Loss=0.5862]


Final Model Tersimpan: best_roberta_clarity_rq4.pth


## STAGE 3 — Extract Predictions ke CSV

In [9]:
final_model.eval()

train_extract_loader = DataLoader(
    SingleTaskDataset(df_train, tokenizer, MAX_LEN, TARGET_TRAIT),
    batch_size=BATCH_SIZE, shuffle=False,
)

def extract_predictions(loader, desc):
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            ids     = batch['input_ids'].to(DEVICE)
            mask    = batch['attention_mask'].to(DEVICE)
            b_score = batch['bertscore'].to(DEVICE).unsqueeze(1)
            tgt     = batch['target'].numpy()

            if DEVICE.type in ["cuda", "xpu"]:
                dtype = torch.bfloat16 if DEVICE.type == "xpu" else torch.float16
                with torch.autocast(device_type=DEVICE.type, dtype=dtype):
                    preds = final_model(ids, mask, b_score)
            else:
                preds = final_model(ids, mask, b_score)

            all_preds.extend(preds.float().cpu().numpy().flatten())
            all_targets.extend(tgt.flatten())
    return all_preds, all_targets

train_preds, train_targets = extract_predictions(train_extract_loader, "Extracting Train Preds")
df_train_out = pd.DataFrame({
    'ground_truth':       train_targets,
    'prediction':         train_preds,
    'rounded_prediction': np.round(np.array(train_preds) * 2) / 2,
})
df_train_out.to_csv(TRAIN_PRED_CSV, index=False)

test_preds, test_targets = extract_predictions(test_loader, "Extracting Test Preds")
df_test_out = pd.DataFrame({
    'ground_truth':       test_targets,
    'prediction':         test_preds,
    'rounded_prediction': np.round(np.array(test_preds) * 2) / 2,
})
df_test_out.to_csv(TEST_PRED_CSV, index=False)

print(f"Train preds -> {TRAIN_PRED_CSV}")
print(f"Test  preds -> {TEST_PRED_CSV}")

Extracting Test Preds: 100%|██████████| 94/94 [00:14<00:00,  6.51it/s]

Train preds -> roberta_train_clarity_rq4.csv
Test  preds -> roberta_test_clarity_rq4.csv


## STAGE 4 — Evaluasi Akhir

In [10]:
pred_classes = np.round(np.array(test_preds)   * 2).astype(int)
true_classes = np.round(np.array(test_targets) * 2).astype(int)
final_qwk    = cohen_kappa_score(true_classes, pred_classes, weights='quadratic')

print("=" * 50)
print(f"QWK FINAL (TEST SET) -- Trait '{TARGET_TRAIT.upper()}': {final_qwk:.4f}")
print("=" * 50)

QWK FINAL (TEST SET) -- Trait 'CLARITY': 0.3499
